### Plot BLASTp scores v. composite scores
### Julian Moran
### 2026-08-28

In [1]:
import boto3
import glob
import logging
import math
import os
import requests
import s3fs
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [5]:
# ============================================================
#       Args
# ============================================================

# API endpoints
ENDPOINT_UNIPROT = "https://rest.uniprot.org/uniprotkb/"
ENDPOINT_UNIPROT_SEARCH = "https://rest.uniprot.org/uniprotkb/search"
ENDPOINT_UNIPROT_MAP = "https://rest.uniprot.org/idmapping"
ENDPOINT_UNIPARC_SEARCH = "https://rest.uniprot.org/uniparc/search"

# MinIO
BUCKET = "iei-project"
PREFIX_GOLD = "03_gold/defense_finder/"
PREFIX_SILVER = "02_silver/defense_finder/"
FILE_COMPOSITE = "composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet"

# Dirs
OUT_DIR_PLOT = f"{REPO_ROOT}/vis/plots"

# Check live objects in MinIO silver
client = boto3.client(
    "s3",
    endpoint_url="http://eagle.tcag.ca:9000",
    aws_access_key_id=MINIO_SECRET,
    aws_secret_access_key=MINIO_KEY,
)
response = client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX_GOLD
)
live_objects = [
    obj["Key"]
    for obj in response.get("Contents", [])
]
live_objects

['03_gold/defense_finder/composite_score.parquet/_SUCCESS',
 '03_gold/defense_finder/composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet',
 '03_gold/defense_finder/defense_human_domain_annotated.parquet',
 '03_gold/defense_finder/final_output_spark.parquet/_SUCCESS',
 '03_gold/defense_finder/final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet',
 '03_gold/defense_finder/griid_gene_subset.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.manifest.json',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/_SUCCESS',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/part-00000-275fe83c-840b-476e-bf63-c47165847863-c000.snappy.parquet']

In [40]:
# ============================================================
#       In
# ============================================================

df_comp_score = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_COMPOSITE}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
)
accessions = df_comp_score["human_entryId"].unique()
logger.info(f"n unique human accessions: {len(accessions)}")
df_comp_score

INFO:__main__:n unique human accessions: 15572


defense_uniprot_ac,human_entryId,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria
str,str,f64,f64,f32,f32,f32,f32,f32
"""A0A5C5QGP9""","""A0A024R9P6""",0.2646,0.000087,null,null,null,null,83.480003
"""A0A2R3IRC4""","""A0A0D9SF92""",0.8315,1.2070e-7,0.93,0.8169,0.1028,83.199997,77.449997
"""A0A4D8PF33""","""A0A140VK70""",0.2883,0.003094,15.01,0.2908,0.1539,80.290001,83.050003
"""A0A7S9D461""","""A0A140VK70""",0.8986,5.6920e-19,2.18,0.535,0.7678,80.290001,89.629997
"""A0A2K9LJD6""","""A0A1B0GVC6""",0.2795,0.007862,10.88,0.2428,0.29,67.610001,91.379997
…,…,…,…,…,…,…,…,…
"""A0A1D7XM13""","""Q9H4E3""",0.6409,0.000001,4.93,0.6556,0.327,84.879997,82.360001
"""A0A7D6CQE3""","""Q9H4E3""",0.5814,0.000001,5.77,0.6733,0.3337,84.879997,75.230003
"""A0A5P3ALB7""","""Q9H4E3""",0.8713,1.2930e-15,2.68,0.739,0.4786,84.879997,89.019997


In [11]:
# ============================================================
#       UniprotKB accessions --> AA seqs
# ============================================================

def get_uniprotkb_seqs(
    accessions: pl.Series,
    uniprotkb_endpoint: str,
) -> pl.DataFrame:
    """
    Submit request for AA sequence for all accessions in `accessions`.
    """

    accessions = accessions.to_list()
    query = " OR ".join(f"accession:{acc}" for acc in accessions)

    response = requests.get(
        uniprotkb_endpoint,
        params={
            "query": query,
            "format": "fasta",
            "size": len(accessions)+400,
        },
    )
    response.raise_for_status()
    records = response.text.strip().split("\n>")
    parsed_accessions = []
    sequences = []

    logger.info(
        f"UniProtKB requested: {len(accessions)}, "
        f"UniProtKB returned: {len(records)}"
    )

    for record in records:
        record = record.lstrip(">")
        lines = record.splitlines()
        header = lines[0]
        sequence = "".join(lines[1:])

        if "|" not in header:
            logger.warning(f"Unexpected FASTA header: {header!r}")
            continue

        parsed_accessions.append(header.split("|")[1])
        sequences.append(sequence)

    return pl.DataFrame({
        "uniprot_accession": parsed_accessions,
        "sequence": sequences,
    })


def get_uniprotkb_seqs_by_batch(
    accessions: pl.Series,
    uniprotkb_endpoint: str,
    batch_size: int = 100,
    wait_time: float = 0.5,
) -> pl.DataFrame:
    """
    Break up request into batches of `batch_size` and call `get_uniprotkb_seqs()` on each batch.
    @user: do not set batch_size > API batch-size limit.
    """
    # Find sequences in UniProtKB
    uniprot_batches = []
    for i in range(0, len(accessions), batch_size):
        batch = accessions[i:i + batch_size]
        uniprot_batches.append(
            get_uniprotkb_seqs(
                accessions=batch,
                uniprotkb_endpoint=uniprotkb_endpoint
            )
        )
        logger.info(
            f"UniProtKB: processed "
            f"{min(i + batch_size, len(accessions))} "
            f"of {len(accessions)} accessions"
        )
        time.sleep(wait_time)
    df_uniprotkb_sequences = pl.concat(uniprot_batches)
    return df_uniprotkb_sequences

accessions = df_comp_score["human_entryId"].unique()
seqs_uniprotkb = get_uniprotkb_seqs_by_batch(
    accessions=accessions,
    uniprotkb_endpoint=ENDPOINT_UNIPROT_SEARCH,
    batch_size=100,
    wait_time=0.5
)
seqs_uniprotkb

INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 56
INFO:__main__:UniProtKB: processed 100 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 65
INFO:__main__:UniProtKB: processed 200 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 53
INFO:__main__:UniProtKB: processed 300 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 71
INFO:__main__:UniProtKB: processed 400 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 69
INFO:__main__:UniProtKB: processed 500 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 70
INFO:__main__:UniProtKB: processed 600 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 71
INFO:__main__:UniProtKB: processed 700 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 72
INFO:__main__:UniProtKB: processed 800 of 15572 accessions
INFO:__main__:UniProtKB 

uniprot_accession,sequence
str,str
"""O94762""","""MSSHHTTFPFDPERRVRSTLKKVFGFDSFK…"
"""Q9H2K8""","""MRKGVLKDPEIADLFYKDDPEELFIGLHEI…"
"""Q8IYB8""","""MSFSRALLWARLPAGRQAGHRAAICSALRP…"
"""Q9H222""","""MGDLSSLTPGGSMGLQVNRGSQSSLEGAPA…"
"""Q12834""","""MAQFAFESDLHSLLQLDAPIPNAPPARWQR…"
…,…
"""E9PCE7""","""MDNFAEGDFTVADYALLEDCPHVDDCVFAA…"
"""Q495V6""","""MVAEVCSMPAASAVKKPFDLRSKMGKWCHH…"
"""A0A384MDT0""","""MTSSDKDFRFMATSDLMSELQKDSIQLDED…"


In [38]:
# ============================================================
#       UniprotKB --> Uniparc
# ============================================================

def map_uniprotkb_to_uniparc(
    accessions: pl.Series,
    uniprot_map_endpoint: str,
    wait_time: float = 0.5
) -> pl.DataFrame:
    """
    Map UniProtKB accessions to UniParc accessions.
    Notably, the UniProt ID mapping endpoint accepts query lengths up to 100,000.
    """
    mapping_response = requests.post(
        f"{uniprot_map_endpoint}/run",
        data={
            "from": "UniProtKB_AC-ID",
            "to": "UniParc",
            "ids": ",".join(accessions.to_list()),
        },
    )
    mapping_response.raise_for_status()
    mapping_job_id = mapping_response.json()["jobId"]
    logger.info(f"Submitted request to {uniprot_map_endpoint}.")
    logger.info("Monitoring for status ...")

    while True:
        status_response = requests.get(
            f"{uniprot_map_endpoint}/status/{mapping_job_id}"
        )
        if not status_response.ok:
            raise RuntimeError(
                f"UniProt ID mapping status request failed: "
                f"{status_response.status_code} {status_response.text}"
            )
        status = status_response.json()
        if status.get("jobStatus") == "RUNNING":
            time.sleep(wait_time)
            continue
        if status.get("jobStatus") == "FAILED":
            raise RuntimeError(
                f"UniProt ID mapping failed: {status}"
            )
        break
    logger.info("Retrieving mapping results ...")

    mapping = []
    url = f"{uniprot_map_endpoint}/uniparc/results/{mapping_job_id}"
    params = {
        "format": "tsv",
        "size": 500,
    }
    while url:
        response = requests.get(url, params=params)
        response.raise_for_status()
        lines = response.text.strip().splitlines()
        if len(lines) > 1:
            mapping.extend(
                line.split("\t")[:2]
                for line in lines[1:]
            )
        url = response.links.get("next", {}).get("url")
        params = None

    df_mapping = pl.DataFrame(
        mapping,
        schema=["uniprot_accession", "uniparc_accession"],
        orient="row",
    )
    logger.info(f"UniProtKB accessions submitted: {len(accessions)}")
    logger.info(f"UniProtKB-to-UniParc mappings returned: {len(df_mapping)}")
    return df_mapping

# Get leftover UniprotKB accessions that still don't have AA seqs
accessions_missing = accessions.filter(
    ~accessions.is_in(seqs_uniprotkb["uniprot_accession"].implode())
)

# Map UniprotKB --> Uniparc
accessions_uniparc = map_uniprotkb_to_uniparc(
    accessions=accessions_missing,
    uniprot_map_endpoint=ENDPOINT_UNIPROT_MAP,
    wait_time=0.5
)
accessions_uniparc = accessions_uniparc.unique("uniprot_accession")
accessions_uniparc

INFO:__main__:Submitted request to https://rest.uniprot.org/idmapping.
INFO:__main__:Monitoring for status ...
INFO:__main__:Retrieving mapping results ...
INFO:__main__:UniProtKB accessions submitted: 5334
INFO:__main__:UniProtKB-to-UniParc mappings returned: 5368


uniprot_accession,uniparc_accession
str,str
"""A0A0E3DCY9""","""UPI00061B8051"""
"""C7FDR2""","""UPI0001B1A0EE"""
"""C0KLQ2""","""UPI0001949372"""
"""F6KRQ2""","""UPI00020D6330"""
"""A0A0E3DDB6""","""UPI00019D9730"""
…,…
"""A0A0E3DC77""","""UPI0000062349"""
"""A0A024R5Z6""","""UPI00001BFAFE"""
"""D6MJE9""","""UPI00001BD8BE"""


In [ ]:
# ============================================================
#       Uniparc --> AA seqs
# ============================================================

def get_uniparc_seqs_by_batch(
    accessions: pl.Series,
    uniparc_endpoint: str,
    batch_size: int = 100,
    wait_time: float = 0.5
) -> pl.DataFrame:

    uniparc_batches = []
    for i in range(0, len(accessions), batch_size):
        batch = accessions[i:i + batch_size]
        query = " OR ".join(f"upi:{accession}" for accession in batch)
        response = requests.get(
            uniparc_endpoint,
            params={
                "query": query,
                "format": "fasta",
                "size": batch_size,
            },
        )
        response.raise_for_status()
        records = response.text.strip().split("\n>")
        logger.info(
            f"Uniparc requested: {len(batch)}, "
            f"Uniparc returned: {len(records)}"
        )
        sequences = []

        for record in records:
            lines = record.lstrip(">").splitlines()
            header = lines[0]
            sequence = "".join(lines[1:])
            upi = header.split()[0]
            sequences.append({
                "uniparc_accession": upi,
                "sequence": sequence,
            })

        uniparc_batches.append(pl.DataFrame(sequences))
        logger.info(
            f"UniParc: processed "
            f"{min(i + batch_size, len(accessions))} "
            f"of {len(accessions)} mapped accessions"
        )

        time.sleep(wait_time)
    return pl.concat(uniparc_batches)

seqs_uniparc = get_uniparc_seqs_by_batch(
    accessions=accessions_uniparc["uniparc_accession"],
    uniparc_endpoint=ENDPOINT_UNIPARC_SEARCH,
    batch_size=100,
    wait_time=0.5
)
seqs_uniparc

INFO:__main__:Uniparc requested: 100, Uniparc returned: 86
INFO:__main__:UniParc: processed 100 of 5334 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 93
INFO:__main__:UniParc: processed 200 of 5334 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 92
INFO:__main__:UniParc: processed 300 of 5334 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 90
INFO:__main__:UniParc: processed 400 of 5334 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 94
INFO:__main__:UniParc: processed 500 of 5334 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 91
INFO:__main__:UniParc: processed 600 of 5334 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 94
INFO:__main__:UniParc: processed 700 of 5334 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 88
INFO:__main__:UniParc: processed 800 of 5334 mapped accessions
INFO:__main__:Uniparc re

uniparc_accession,sequence
str,str
"""UPI000008A0FF""","""SHSMRYFYTAVSRPGRGEPHFIAVGYVDDT…"
"""UPI000008A34A""","""SHSMRYFSTSVSRPGSGEPRFIAVGYVDDT…"
"""UPI00004CCEE7""","""SHSMRYFYTAMSRPGRGEPRFIAVGYVDDT…"
"""UPI00017FB904""","""SHSMRYFSTSVSRPGSGEPRFIAVGYVDDT…"
"""UPI0000072115""","""SHSMRYFYTAMSRPGRGEPRFIAVGYVDDT…"
…,…
"""UPI00027EE881""","""SHSMRYFYTNVSRPGRGEPHFIAVGYVDDT…"
"""UPI000327609B""","""SHSMRYFYTAMSRPGRGEPRFIAVGYVDDT…"
"""UPI000655AA9F""","""SHSMRYFYTAMSRPGRGEPRFIAVGYVDDT…"


In [44]:
# ============================================================
#       Wrangle
# ============================================================

seqs = pl.concat([
    seqs_uniprotkb,
    (
        accessions_uniparc
        .join(
            seqs_uniparc.unique("uniparc_accession"),
            on="uniparc_accession",
            how="left"
        )
        .select(["uniprot_accession", "sequence"])
    )
]).drop_nulls()
seqs

uniprot_accession,sequence
str,str
"""O94762""","""MSSHHTTFPFDPERRVRSTLKKVFGFDSFK…"
"""Q9H2K8""","""MRKGVLKDPEIADLFYKDDPEELFIGLHEI…"
"""Q8IYB8""","""MSFSRALLWARLPAGRQAGHRAAICSALRP…"
"""Q9H222""","""MGDLSSLTPGGSMGLQVNRGSQSSLEGAPA…"
"""Q12834""","""MAQFAFESDLHSLLQLDAPIPNAPPARWQR…"
…,…
"""A0A0E3DC77""","""SHSMRYFYTAVSRPGRGEPRFIAVGYVDDT…"
"""A0A024R5Z6""","""MWKRSEQMKIKSGKCNMAAAMETEQLGVEI…"
"""D6MJE9""","""SHSMRYFYTAMSRPGRGEPRFISVGYVDDT…"
